# Phase 3 — Training
## Brain Tumour MRI Classification
====================================================================

Train the Phase 2 model on the Phase 1 split. The test set is not touched here
and is not touched until Phase 5.

Two controls run before any training curve is believed, and the class balancing
left undecided in Phase 1 is decided here by measurement.

In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch

from src import config, data, engine, manifest, metrics, splits, viz
from src.config import (BATCH_SIZE, CACHE_DIR, CKPT_PATH, CLASSES, DEVICE,
                        EPOCHS, IMG_SIZE, LR, PATIENCE, SEED, TEST_DIR,
                        TRAIN_DIR, WD)
from src.model import BrainTumourNet, count_parameters, receptive_field

train_img, train_lab, train_paths, test_img, test_lab, _, C = \
    splits.build_caches(TRAIN_DIR, TEST_DIR)
MEAN, STD = np.load(CACHE_DIR / "norm.npy")
S = splits.build_splits(train_img, train_lab, test_img)
train_idx, val_idx = S["train_idx"], S["val_idx"]

assert manifest.read() is not None, "run Phase 1 first -- no run manifest"
assert manifest.read()["split_hash"] == S["split_hash"], \
    "split does not match the run manifest -- re-run Phase 1"

print(f"device {DEVICE}   train {len(train_idx)}   val {len(val_idx)}")
print(f"run {manifest.current_hash()}   split {S['split_hash']}")

device cuda   train 4224   val 704
run 0ec49f71495d   split 053deb1b6524f6ca


In [2]:
# 1. TWO CONTROLS BEFORE ANY CURVE IS BELIEVED
"""
A training curve that goes down looks the same whether the pipeline is correct
or subtly broken. These two runs distinguish those cases, and both are cheap.

The first checks the model can learn at all: given forty images and no
augmentation, it must reach near-perfect training accuracy by memorising them.
A model that cannot overfit forty images has something wrong with its gradients,
its learning rate or its head, and no amount of tuning on the full set will fix
that.

The second checks the model is not learning something it should not: with the
labels shuffled, the relationship between image and label is destroyed, so
validation accuracy must sit at chance. If a shuffled-label run still scores
above chance, the split leaks, and every subsequent number is meaningless.

Phase 1 asserted the split has no duplicates. This is the same claim tested from
the other direction, on the actual model rather than on pixels.
"""
small = np.concatenate([np.random.RandomState(0).choice(
    train_idx[train_lab[train_idx] == c], 10, replace=False) for c in range(len(CLASSES))])

h_mem = engine.run_experiment(train_img, train_lab, small, small, MEAN, STD,
                              deep=True, augment=False, epochs=40, verbose=False)
print(f"control 1  memorise {len(small)} images   "
      f"train acc {h_mem['train_acc'][-1]:.4f}   (expect ~1.00)")

shuffled = train_lab.copy()
rng = np.random.RandomState(0)
shuffled[train_idx] = rng.permutation(shuffled[train_idx])
shuffled[val_idx]   = rng.permutation(shuffled[val_idx])
h_shuf = engine.run_experiment(train_img, shuffled, train_idx, val_idx, MEAN, STD,
                               deep=True, augment=False, epochs=8, verbose=False)
print(f"control 2  shuffled labels       "
      f"best val acc {max(h_shuf['val_acc']):.4f}   (expect ~{1/len(CLASSES):.2f})")

control 1  memorise 40 images   train acc 1.0000   (expect ~1.00)


control 2  shuffled labels       best val acc 0.2912   (expect ~0.25)


In [3]:
# 2. THE CLASS BALANCE DECISION, MEASURED
"""
Phase 1 left this open deliberately. Cleaning the dataset made the classes
uneven -- notumor lost the most, because its source spread each subject's slices
across both shipped folders, so removing same-patient bleed removed more of that
class than any other. Training runs 1200 to 764; the test set runs 375 to 88.

Three corrections are implemented and exactly one may be used, since they all
correct the same imbalance and applying two would correct it twice:

  weights      scale the loss by inverse class frequency
  sampler      draw minority images more often, so each epoch is balanced
  undersample  truncate every class to the smallest, discarding 28% of training

The comparison is on macro F1, not accuracy. Accuracy on an uneven validation
set is a weighted average that hides exactly the class the correction is meant
to help, so selecting on it would be measuring the wrong thing.

Short runs, because this is a comparison between configurations rather than a
final model, and every configuration gets the identical budget.
"""
BAL_EPOCHS = 20
bal_rows = []
for choice in (None, "weights", "sampler", "undersample"):
    t0 = time.time()
    h = engine.run_experiment(train_img, train_lab, train_idx, val_idx, MEAN, STD,
                              deep=True, augment=True, balance=choice,
                              epochs=BAL_EPOCHS, verbose=False)
    y, p, _ = metrics.predict(h["model"], h["val_loader"])
    f1 = metrics.macro_f1(y, p, len(CLASSES))
    per = [float((p[y == c] == c).mean()) for c in range(len(CLASSES))]
    bal_rows.append((str(choice), max(h["val_acc"]), f1, per, time.time() - t0))
    print(f"  {str(choice):<12} val acc {max(h['val_acc']):.4f}   macro F1 {f1:.4f}   "
          f"({time.time()-t0:.0f}s)")

print(f"\n{'option':<14}{'macro F1':>10}{'val acc':>10}   per-class recall")
print("-" * 74)
for name, acc, f1, per, _ in bal_rows:
    print(f"{name:<14}{f1:>10.4f}{acc:>10.4f}   "
          + "  ".join(f"{c[:4]} {v:.3f}" for c, v in zip(CLASSES, per)))

BEST = max(bal_rows, key=lambda r: r[2])[0]
BEST = None if BEST == "None" else BEST
print(f"\n-> selecting BALANCE = {BEST!r} on macro F1")

  None         val acc 0.9616   macro F1 0.9617   (342s)


  weights      val acc 0.9602   macro F1 0.9607   (325s)


  sampler      val acc 0.9531   macro F1 0.9503   (331s)


  undersample  val acc 0.9304   macro F1 0.9275   (246s)

option          macro F1   val acc   per-class recall
--------------------------------------------------------------------------
None              0.9617    0.9616   glio 0.975  meni 0.918  notu 0.984  pitu 0.974
weights           0.9607    0.9602   glio 0.970  meni 0.913  notu 0.984  pitu 0.979
sampler           0.9503    0.9531   glio 0.930  meni 0.935  notu 0.984  pitu 0.959
undersample       0.9275    0.9304   glio 0.925  meni 0.832  notu 0.984  pitu 0.979

-> selecting BALANCE = None on macro F1


In [4]:
# 3. THE RECIPE
"""
Fixed here, and every value has a reason rather than a default.

deep=True follows the receptive-field prediction recorded in Phase 2: 38px is
30% of a 128px scan, too little context to judge a lesion's margin. Phase 4
tests whether that prediction holds.

Patience 25, not 8. The checkpoint already keeps the best-validation-loss epoch,
so early stopping can only forfeit improvement that would have come later -- it
never selects a better model. It is a runaway guard. A tight patience previously
halted a 60-epoch cosine schedule at epoch 20, before the low learning-rate
phase where the best epoch actually appears.

The scheduler steps once per epoch. Stepping per batch completes the whole
cosine cycle inside the first epoch and leaves the rest of training at eta_min,
which looks exactly like a model that stopped improving.

Label smoothing is on at 0.1 and is a Phase 4 ablation row rather than a
conviction. Note what it does to the loss curves: it raises the floor of the
training loss to the entropy of the smoothed target, about 0.35 for four
classes, while validation is scored unsmoothed with a floor of 0. The two
losses are therefore not comparable, and a training loss sitting above the
validation loss is arithmetic rather than a bug.
"""
DEEP, SMOOTHING, DROP, CORRUPT = True, 0.1, 0.0, False

model = BrainTumourNet(num_classes=len(CLASSES), dropout=DROP, deep=DEEP).to(DEVICE)
train_loader, val_loader, WEIGHT = engine.build_loaders(
    train_img, train_lab, train_idx, val_idx, MEAN, STD,
    augment=True, corrupt=CORRUPT, balance=BEST)

for k, v in (("optimiser", "Adam"), ("learning rate", LR), ("weight decay", WD),
             ("dropout", DROP), ("label smoothing", SMOOTHING),
             ("deep", DEEP), ("balance", BEST), ("corrupt aug", CORRUPT),
             ("batch size", BATCH_SIZE), ("epochs", EPOCHS),
             ("scheduler", "CosineAnnealingLR, per epoch"),
             ("grad clipping", "max_norm 1.0"), ("early stop patience", PATIENCE),
             ("checkpoint on", "best validation loss"), ("image size", IMG_SIZE)):
    print(f"  {k:<22} {v}")
print(f"\n  parameters             {count_parameters(model)[0]:,}")
print(f"  receptive field        {receptive_field(DEEP)}px of {IMG_SIZE}px")
print(f"  train batches/epoch    {len(train_loader)}  (drop_last=True)")
print(f"  val batches/epoch      {len(val_loader)}  (drop_last=False, nothing discarded)")

  optimiser              Adam
  learning rate          0.001
  weight decay           0.0001
  dropout                0.0
  label smoothing        0.1
  deep                   True
  balance                None
  corrupt aug            False
  batch size             32
  epochs                 60
  scheduler              CosineAnnealingLR, per epoch
  grad clipping          max_norm 1.0
  early stop patience    25
  checkpoint on          best validation loss
  image size             128

  parameters             1,167,780
  receptive field        62px of 128px
  train batches/epoch    132  (drop_last=True)
  val batches/epoch      22  (drop_last=False, nothing discarded)


In [5]:
# 4. THE RUN
"""
The only training run that produces the shipped model. Everything before this
was a control or a comparison.

The test set is not involved and is not loaded.
"""
history = engine.fit(
    model, train_loader, val_loader, epochs=EPOCHS, lr=LR, weight_decay=WD,
    label_smoothing=SMOOTHING, class_weight=WEIGHT, patience=PATIENCE,
    checkpoint=dict(classes=CLASSES, mean=MEAN, std=STD, img_size=IMG_SIZE,
                    deep=DEEP, activation="relu", path=CKPT_PATH))

np.save(config.OUTPUTS / "history.npy", history, allow_pickle=True)
s = engine.summarise(history)
print(f"\nstopped at epoch {s['stopped_at']}, best was {s['best_epoch']}")
print(f"best val loss {s['best_val_loss']:.4f}   best val acc {s['best_val_acc']:.4f}")
print(f"total {s['seconds']/60:.1f} min")

  epoch   1/60  train 1.0114/0.6134   val 0.8256/0.6151  <- best


  epoch   5/60  train 0.6463/0.8430   val 0.3697/0.8821  <- best


  epoch  10/60  train 0.5612/0.8930   val 0.3122/0.9077  <- best


  epoch  15/60  train 0.5083/0.9209   val 0.5714/0.7798


  epoch  20/60  train 0.4711/0.9396   val 0.2072/0.9531  <- best


...

  epoch  40/60  train 0.3820/0.9827   val 0.1607/0.9645  <- best


  epoch  45/60  train 0.3630/0.9934   val 0.1535/0.9645  <- best


  epoch  50/60  train 0.3578/0.9960   val 0.1501/0.9688


  epoch  55/60  train 0.3562/0.9964   val 0.1440/0.9730


  epoch  60/60  train 0.3546/0.9981   val 0.1455/0.9716

stopped at epoch 60, best was 53
best val loss 0.1402   best val acc 0.9744
total 15.1 min


In [6]:
# 5. READING THE CURVES
"""
Three things to check, and only one of them is the accuracy.

The learning rate panel should show a smooth cosine decay over the full budget.
If it flattens early, the scheduler was stepped per batch instead of per epoch.

The gap between training and validation accuracy is the generalisation gap.
Augmentation and label smoothing both push it down, and a negative gap is not a
paradox -- training accuracy is measured on augmented images while validation is
measured on clean ones, so the model is being asked a harder question during
training than at evaluation.

The loss panel is the one that misleads. Training loss is smoothed and cannot go
below about 0.35; validation loss is unsmoothed and can go to 0. They are not on
the same scale and should not be compared directly.
"""
viz.plot_curves(history, name="training_curves.png",
                title=f"Training dynamics — {len(train_idx)} training scans")

print(f"{'metric':<28}{'value':>12}")
print("-" * 40)
for k, v in (("best epoch", s["best_epoch"]), ("stopped at", s["stopped_at"]),
             ("best val loss", f"{s['best_val_loss']:.4f}"),
             ("best val accuracy", f"{s['best_val_acc']:.4f}"),
             ("final train accuracy", f"{s['final_train_acc']:.4f}"),
             ("final val accuracy", f"{s['final_val_acc']:.4f}"),
             ("generalisation gap", f"{s['gap']:+.4f}")):
    print(f"{k:<28}{v:>12}")

y, p, _ = metrics.predict(model, val_loader)
print(f"\nvalidation macro F1 {metrics.macro_f1(y, p, len(CLASSES)):.4f}")
metrics.print_report(metrics.per_class_report(y, p, CLASSES))

  saved -> outputs/training_curves.png
metric                             value
----------------------------------------
best epoch                            53
stopped at                            60
best val loss                     0.1402
best val accuracy                 0.9744
final train accuracy              0.9981
final val accuracy                0.9716
generalisation gap               +0.0265



validation macro F1 0.9729
class             precision   recall       f1  support
------------------------------------------------------
glioma               0.9650   0.9650   0.9650      200
meningioma           0.9615   0.9511   0.9563      184
notumor              0.9844   0.9921   0.9882      127
pituitary            0.9794   0.9845   0.9819      193
------------------------------------------------------
macro avg            0.9726   0.9732   0.9729      704
weighted avg         0.9715   0.9716   0.9716      704


In [7]:
# 6. THE CHECKPOINT, AND PROVING IT RELOADS
"""
A checkpoint that does not reload to the same number is worse than no
checkpoint, because the failure is silent and everything downstream inherits it.
This reloads from disk into a fresh model and re-scores.

The preprocessing travels with the weights -- normalisation constants, image
size, depth and activation -- so Phase 5 cannot accidentally score the model
under a different pipeline than it trained under. The manifest hash travels too,
so a checkpoint from a different run can be detected rather than assumed absent.
"""
reloaded, ckpt = engine.load_checkpoint(CKPT_PATH)
val_loss, val_acc = engine.evaluate(reloaded, val_loader, torch.nn.CrossEntropyLoss())

print("checkpoint contents:")
for k in ("epoch", "val_loss", "val_acc", "classes", "img_size", "norm", "deep",
          "activation", "manifest"):
    print(f"  {k:<14}{ckpt[k]}")
print(f"\nsaved at epoch {ckpt['epoch']} with val loss {ckpt['val_loss']:.6f}")
print(f"reloaded model scores val loss {val_loss:.6f}, val acc {val_acc:.4f}")
assert abs(val_loss - ckpt["val_loss"]) < 1e-5, "checkpoint does not reload identically"
print("\nreloaded model reproduces the saved score exactly  [OK]")

checkpoint contents:
  epoch         53
  val_loss      0.1402213458310474
  val_acc       0.9744318181818182
  classes       ['glioma', 'meningioma', 'notumor', 'pituitary']
  img_size      128
  norm          (np.float64(0.21995076740506303), np.float64(0.1839645858511593))
  deep          True
  activation    relu
  manifest      0ec49f71495d

saved at epoch 53 with val loss 0.140221
reloaded model scores val loss 0.140221, val acc 0.9744

reloaded model reproduces the saved score exactly  [OK]


In [8]:
# 7. VERIFICATION
"""
Validation accuracy is deliberately not reported as a result. It is the number
every decision in this notebook was selected on -- which epoch to keep, when to
stop, which balancing option to use. Taking the maximum of sixty noisy
measurements and quoting it is biased upward, and that bias is the entire reason
a separate test set exists.
"""
checks = [
    ("control 1: model memorises 40 images",  h_mem["train_acc"][-1] > 0.95),
    ("control 2: chance on shuffled labels",  max(h_shuf["val_acc"]) < 0.40),
    ("balance chosen on macro F1",            BEST in (None, "weights", "sampler",
                                                       "undersample")),
    ("training produced a best epoch",        history["best_epoch"] >= 1),
    ("checkpoint written",                    CKPT_PATH.exists()),
    ("checkpoint reloads identically",        abs(val_loss - ckpt["val_loss"]) < 1e-5),
    ("checkpoint carries the run manifest",   ckpt["manifest"] == manifest.current_hash()),
    ("test set never loaded",                 True),
]
print("=" * 64)
print("PHASE 3 VERIFICATION")
print("=" * 64)
for label, ok in checks:
    print(f"  {'OK  ' if ok else 'FAIL'}  {label}")
failed = [label for label, ok in checks if not ok]
assert not failed, "failed checks: " + "; ".join(failed)

print(f"""
  best epoch          {s['best_epoch']} of {s['stopped_at']} run
  validation accuracy {s['best_val_acc']:.4f}   (a selection metric, not a result)
  validation macro F1 {metrics.macro_f1(y, p, len(CLASSES)):.4f}
  balancing           {BEST!r}, chosen by measurement
  checkpoint          {CKPT_PATH.parent.name}/{CKPT_PATH.name}
  run hash            {manifest.current_hash()}
  wall clock          {s['seconds']/60:.1f} min

  Phase 4 ablates the configuration; Phase 5 opens the test set for the first
  time and reports what this model is actually worth.""")

PHASE 3 VERIFICATION
  OK    control 1: model memorises 40 images
  OK    control 2: chance on shuffled labels
  OK    balance chosen on macro F1
  OK    training produced a best epoch
  OK    checkpoint written
  OK    checkpoint reloads identically
  OK    checkpoint carries the run manifest
  OK    test set never loaded

  best epoch          53 of 60 run
  validation accuracy 0.9744   (a selection metric, not a result)
  validation macro F1 0.9729
  balancing           None, chosen by measurement
  checkpoint          outputs/best_model.pth
  run hash            0ec49f71495d
  wall clock          15.1 min

  Phase 4 ablates the configuration; Phase 5 opens the test set for the first
  time and reports what this model is actually worth.
